# Feature Track 3: Query Improvement Methods

# Setup

In [5]:
from loguru import logger
import sys

from conversational_toolkit.embeddings.openai import OpenAIEmbeddings

from conversational_toolkit.agents.base import QueryWithContext

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
    build_llm,
    build_agent,
    SYSTEM_PROMPT,
    EMBEDDING_MODEL,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")  # hides DEBUG

2

In [6]:
chunks = load_chunks(5)

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
vector_store = await build_vector_store(chunks, embedding_model, reset=False)
llm = build_llm("openai", model_name="gpt-4o-mini")

2026-03-05 15:40:54.787 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:201 - Skipping unsupported file type '': .DS_Store


5


In [7]:
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,
)

# Query Expansion

See conversational_toolkit.utils.retriever.py -> query_expansion and reciprocal_rank_fusion

In [8]:
logger.add(sys.stderr, level="DEBUG")

3

In [9]:
agent_query_expansion = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=5,
)

2026-03-05 15:41:00.598 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_agent:347 - RAG agent ready (top_k=5  query_expansion=5)


In [10]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

2026-03-05 15:41:01.913 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)


Answer:
[MessageContent(type='text', text='The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Product 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)]
Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'


In [11]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent_query_expansion.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

2026-03-05 15:41:08.843 | DEBUG    | conversational_toolkit.llms.openai:generate:114 - Completion: ChatCompletion(id='chatcmpl-DG4AtpRqb1Rxveu3IfgjYKypAxWHY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='1. Which pallets in our inventory have third-party verified Environmental Product Declarations?\n2. How to find pallets with verified EPD in our product range?\n3. List of pallets with third-party EPD certification in our portfolio.\n4. What are the benefits of third-party verified EPD for pallets?\n5. How to verify if a pallet has a third-party EPD?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1772721667, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_373a14eb6f', usage=CompletionUsage(completion_tokens=76, prompt_tokens=103, total_tokens=179, completion_tokens_details=CompletionTokensDetails(accepted_predict

Answer:
[MessageContent(type='text', text='The pallets in your portfolio that have a third-party verified Environmental Product Declaration (EPD) are:\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Products 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers are compliant with the requirement of having EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)]
Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.5 FSC and Other Chain-of-Custody Certifications'


# Hallucinate answer

In [12]:
agent_hyde = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,
    enable_hyde=True,
)

2026-03-05 15:41:18.186 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_agent:347 - RAG agent ready (top_k=5  query_expansion=0)


In [14]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("Answer:")
print(f"{response.content[0].text}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

2026-03-05 15:41:34.865 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)


Answer:
The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):

1. **IPG** - Products 50-100, 50-101
2. **CPR System** - Product 32-100
3. **Relicyc** - Product 32-103
4. **StabilPlastik** - Product 32-105
5. **Redbox** - Product 11-100
6. **Grupak** - Product 11-101

These suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).
Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'


In [15]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent_hyde.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

2026-03-05 15:41:45.472 | DEBUG    | conversational_toolkit.llms.openai:generate:114 - Completion: ChatCompletion(id='chatcmpl-DG4BTNFkxJx33emSLn7oNsVwt1w2f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='To identify which pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD), you would need to review the documentation for each pallet product. Look for EPD labels or certificates that indicate third-party verification, typically provided by recognized organizations. This information is often available in product specifications or sustainability reports from your suppliers.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1772721703, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_a1ddba3226', usage=CompletionUsage(completion_tokens=65, prompt_tokens=63, total_tokens=128, comple

Answer:
[MessageContent(type='text', text='The following pallets in your portfolio have a third-party verified Environmental Product Declaration (EPD):\n\n1. **IPG** - Products 50-100, 50-101\n2. **CPR System** - Product 32-100\n3. **Relicyc** - Product 32-103\n4. **StabilPlastik** - Product 32-105\n5. **Redbox** - Product 11-100\n6. **Grupak** - Product 11-101\n\nThese suppliers have compliant EPDs on file as of January 2025 (source: ce19759d-d19a-4b36-bf4b-a34eac175a8f).', image_url=None)]
Sources (5):
  'ART_internal_procurement_policy.pdf'  |  '### 3.2 Environmental Product Declarations (EPD)'
  'ART_internal_procurement_policy.pdf'  |  '## 2. Evidence Standards'
  'ART_internal_procurement_policy.pdf'  |  '### 3.4 Recycled Content Claims'
  'ART_internal_procurement_policy.pdf'  |  '### 3.1 All Suppliers and Products'
  'ART_internal_procurement_policy.pdf'  |  '### 3.3 PFAS (Per- and Polyfluoroalkyl Substances)'


---------------------------